# Modbus/TCP PCAP Feature Extraction for Anomaly Detection

This notebook extracts Modbus/TCP fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection.

In [21]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [1]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os
import numpy as np

In [23]:
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """
    Extract Modbus/TCP fields from a PCAP file using tshark.
    
    Args:
        pcap_path: Path to the PCAP file
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract (mbtcp = Modbus/TCP header, modbus = PDU)
    fields = [
        "frame.number",
        "frame.time_epoch",        
        "frame.len",                # Total frame length                   ANOMALOUYS?
        "_ws.col.protocol",         # Protocols in frame
        "ip.src",
        "ip.dst",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",                  # TCP segment length                   ANOMALOUYS?
        "mbtcp.trans_id",           # Transaction Identifier               ANOMALOUYS?
        "mbtcp.prot_id",
        "mbtcp.len",                # Length of remaining bytes in PDU     ANOMALOUYS?
        "mbtcp.unit_id",            # Unit Identifier                      ANOMALOUYS?
        "modbus.func_code",         # Function code                        ANOMALOUYS?
        "modbus.reference_num",     # Reference number (address)           ANOMALOUYS?
        "modbus.word_cnt",          # Number of data words in PDU          ANOMALOUYS?
        "modbus.bit_cnt",           # Number of data bits in PDU
        "modbus.byte_cnt",          # Number of data bytes in PDU          ANOMALOUYS?
        "modbus.exception_code",    # Exception code if present            ANOMALOUYS?
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", pcap_path,  
        "-T", "fields",
        "-Y", "modbus",  # Uncomment to filter only Modbus packets
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]

    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    # Process each line
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'protocols': parse_str(row.get('_ws.col.protocol')),
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'src_port': parse_int(row.get('tcp.srcport')),
            'dst_port': parse_int(row.get('tcp.dstport')),
            'tcp_len': parse_int(row.get('tcp.len')),
            'transaction_id': parse_int(row.get('mbtcp.trans_id')),
            'protocol_id': parse_int(row.get('mbtcp.prot_id')),
            'modbus_length': parse_int(row.get('mbtcp.len')),
            'unit_id': parse_int(row.get('mbtcp.unit_id')),
            'function_code': parse_int(row.get('modbus.func_code')),
            'reference_num': parse_int(row.get('modbus.reference_num')),
            'word_count': parse_int(row.get('modbus.word_cnt')),
            'bit_count': parse_int(row.get('modbus.bit_cnt')),
            'byte_count': parse_int(row.get('modbus.byte_cnt')),
            'exception_code': parse_int(row.get('modbus.exception_code')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # Derive additional features
        record['is_request'] = 1 if record['dst_port'] == 502 else 0
        record['is_response'] = 1 if record['src_port'] == 502 else 0
        record['is_exception'] = 1 if record['exception_code'] is not None else 0
        
        records.append(record)
    
    return records

In [24]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names."""
    df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [25]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    import numpy as np
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Rolling statistics (last 10 packets)
    df['rolling_iat_mean'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).mean()
    df['rolling_iat_std'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).std()
    
    # Requests per second (rolling 1-second window) - optimized with searchsorted
    timestamps = df['timestamp'].values
    is_request = df['is_request'].values
    
    # Cumulative request count for efficient range queries
    cumulative_requests = np.cumsum(is_request)
    
    # For each packet, count requests in [timestamp - 1, timestamp]
    requests_per_second = np.zeros(len(df), dtype=int)
    for i, ts in enumerate(timestamps):
        # Find index of first packet within the 1-second window
        start_idx = np.searchsorted(timestamps, ts - 1, side='left')
        # Count requests from start_idx to i (inclusive)
        if start_idx > 0:
            requests_per_second[i] = cumulative_requests[i] - cumulative_requests[start_idx - 1]
        else:
            requests_per_second[i] = cumulative_requests[i]
    
    df['requests_per_second'] = requests_per_second
    
    return df

In [ ]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df


def add_rtt_feature(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add round trip time (RTT) by matching Modbus requests with responses.
    
    RTT is calculated as the time between a request and its corresponding response,
    matched by transaction_id and IP address pairs.
    """
    
    
    df = df.sort_values('timestamp').reset_index(drop=True)
    df['round_trip_time'] = np.nan
    
    # Dictionary to track pending requests: (transaction_id, client_ip, server_ip) -> (index, timestamp)
    pending_requests = {}
    
    for idx, row in df.iterrows():
        trans_id = row['transaction_id']
        
        if row['is_request'] == 1:
            # This is a request (going to port 502)
            # Key: (transaction_id, client_ip, server_ip)
            key = (trans_id, row['src_ip'], row['dst_ip'])
            pending_requests[key] = (idx, row['timestamp'])
            
        elif row['is_response'] == 1:
            # This is a response (coming from port 502)
            # Key: (transaction_id, client_ip, server_ip) - note src/dst are swapped
            key = (trans_id, row['dst_ip'], row['src_ip'])
            
            if key in pending_requests:
                req_idx, req_timestamp = pending_requests[key]
                rtt = row['timestamp'] - req_timestamp
                
                # Set RTT on both the response and the original request
                df.at[idx, 'round_trip_time'] = rtt
                df.at[req_idx, 'round_trip_time'] = rtt
                
                # Remove the matched request
                del pending_requests[key]
    
    return df

## Usage Example

In [27]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
relative_path = "data\\raw\\captures1_v2\\mitm\\eth2dump-mitm-change-5m-1h_1.pcap"
absolute_path = (Path("../..") / relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = absolute_path  # Update this path
OUTPUT_CSV = "modbus_features.csv"

Using PCAP file at: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-5m-1h_1.pcap


In [28]:
# Extract raw Modbus features
print(f"Processing: {PCAP_PATH}")
records = extract_modbus_features(PCAP_PATH)
print(f"Extracted {len(records)} Modbus packets")

Processing: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-5m-1h_1.pcap
Extracted 23283 Modbus packets


In [29]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = add_temporal_features(df)
    df = add_flow_features(df)
    df = add_rtt_feature(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")


DataFrame shape: (23283, 33)

Columns: ['packet_number', 'timestamp', 'datetime', 'protocols', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'tcp_len', 'transaction_id', 'protocol_id', 'modbus_length', 'unit_id', 'function_code', 'reference_num', 'word_count', 'bit_count', 'byte_count', 'exception_code', 'pkt_len', 'is_request', 'is_response', 'is_exception', 'function_name', 'inter_arrival_time', 'time_from_start', 'rolling_iat_mean', 'rolling_iat_std', 'requests_per_second', 'flow_id', 'flow_pkt_count', 'unique_func_codes', 'rtt']


In [30]:
# Preview the data
df.head(10)

,packet_number,timestamp,datetime,protocols,src_ip,dst_ip,src_port,dst_port,tcp_len,transaction_id,...,function_name,inter_arrival_time,time_from_start,rolling_iat_mean,rolling_iat_std,requests_per_second,flow_id,flow_pkt_count,unique_func_codes,rtt
0,2,1.535066e+09,2018-08-23T19:15:55.134133,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,Read Holding Registers,NaN,0.000000,NaN,NaN,1,172.27.224.250_172.27.224.70,1,1.0,0.005753
1,3,1.535066e+09,2018-08-23T19:15:55.139886,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,Read Holding Registers,0.005753,0.005753,0.005753,NaN,1,172.27.224.250_172.27.224.70,2,1.0,0.005753
2,5,1.535066e+09,2018-08-23T19:15:55.446305,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,Read Holding Registers,0.306419,0.312172,0.156086,0.212603,2,172.27.224.250_172.27.224.70,3,1.0,0.002931
3,6,1.535066e+09,2018-08-23T19:15:55.449236,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,Read Holding Registers,0.002931,0.315103,0.105034,0.174410,2,172.27.224.250_172.27.224.70,4,1.0,0.002931
4,8,1.535066e+09,2018-08-23T19:15:55.758230,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,Read Holding Registers,0.308994,0.624097,0.156024,0.175155,3,172.27.224.250_172.27.224.70,5,1.0,0.011681
5,9,1.535066e+09,2018-08-23T19:15:55.769911,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,Read Holding Registers,0.011681,0.635778,0.127156,0.164853,3,172.27.224.250_172.27.224.70,6,1.0,0.011681
6,11,1.535066e+09,2018-08-23T19:15:56.070166,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,Read Holding Registers,0.300255,0.936033,0.156006,0.163508,4,172.27.224.250_172.27.224.70,7,1.0,0.009093
7,12,1.535066e+09,2018-08-23T19:15:56.079259,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,Read Holding Registers,0.009093,0.945126,0.135018,0.159256,4,172.27.224.250_172.27.224.70,8,1.0,0.009093
8,15,1.535066e+09,2018-08-23T19:15:56.382174,Modbus/TCP,172.27.224.70,172.27.224.250,49499,502,12,0,...,Read Holding Registers,0.302915,1.248041,0.156005,0.158943,4,172.27.224.250_172.27.224.70,9,1.0,0.007769
9,16,1.535066e+09,2018-08-23T19:15:56.389943,Modbus/TCP,172.27.224.250,172.27.224.70,502,49499,31,0,...,Read Holding Registers,0.007769,1.255810,0.139534,0.156674,4,172.27.224.250_172.27.224.70,10,1.0,0.007769


In [31]:
# Summary statistics
print("Function Code Distribution:")
print(df['function_name'].value_counts())
print("\nBasic Statistics:")
df[['pkt_len', 'inter_arrival_time', 'modbus_length']].describe()

Function Code Distribution:
function_name
Read Holding Registers    22041
Write Single Register      1242
Name: count, dtype: int64

Basic Statistics:


,pkt_len,inter_arrival_time,modbus_length
count,23283.000000,23282.000000,23283.000000
mean,75.018168,0.154607,15.004252
std,9.600170,0.196350,9.487260
min,66.000000,0.000047,6.000000
25%,66.000000,0.007161,6.000000
50%,66.000000,0.076889,6.000000
75%,85.000000,0.304202,25.000000
max,270.000000,2.901330,25.000000


In [32]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")


Saved to: modbus_features.csv


## Feature Summary for Anomaly Detection

| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Protocol** | `function_code`, `unit_id`, `transaction_id` | Detect unauthorized commands |
| **Payload** | `reference_num`, `word_count`, `byte_count`, `reg_value` | Detect data manipulation |
| **Temporal** | `inter_arrival_time`, `rolling_iat_*`, `rtt` | Detect timing anomalies, DoS, network issues |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |
| **Error** | `is_exception`, `exception_code` | Detect probing, fuzzing |